# 🚀 MasterFabric Academy - Google Colab PEFT Fine-Tuning & Model Merging

Bu notebook, yerel makinenizi yormadan ve ısıtmadan Google Colab'ın ücretsiz **Tesla T4 GPU (16 GB VRAM)** altyapısını kullanarak Gemma modelinize ince ayar (fine-tuning) yapmanızı, ardından eğittiğiniz LoRA adaptörünü ana model ile birleştirip (merge) tam bir model olarak Hugging Face profilinize yüklemenizi sağlar.

---

### ⚠️ ÖNEMLİ ADIMLAR (Erişim & GPU):
1. **Model İzni:** `google/gemma-2b-it` korumalı (gated) bir modeldir. Eğer henüz yapmadıysanız, [Hugging Face Gemma-2b-it Model Kartı](https://huggingface.co/google/gemma-2b-it) sayfasına gidip lisans sözleşmesini kabul etmeniz gerekir.
2. **GPU Aktivasyonu:** Üst menüden **Runtime** (Çalışma Zamanı) -> **Change runtime type** (Çalışma zamanı türünü değiştir) seçeneğine tıklayın. Donanım hızlandırıcı olarak **T4 GPU** seçin ve kaydedin.

---

### 📂 Dosya Yükleme:
Sol taraftaki klasör simgesine (Files) tıklayın ve yerel bilgisayarınızdan şu 3 dosyayı sürükleyip buraya yükleyin:
1. `journal_finetune_dataset.jsonl` (Eğitim veri seti)
2. `peft_finetune.py` (Eğitim scripti)
3. `merge_and_push.py` (Model birleştirme ve yükleme scripti)

### 1. Bağımlılıkları Yükleme
İnce ayar ve birleştirme işlemleri için gerekli olan PyTorch, Transformers, PEFT ve Accelerate kütüphanelerini kuralım. Ayrıca Colab'daki uyumsuz `torchao` kütüphanesini güncelliyoruz:

In [ ]:
!pip install torch transformers peft bitsandbytes accelerate huggingface_hub psutil requests
!pip install "torchao>=0.16.0" -U

### 2. LoRA Eğitimini Başlatma
Colab T4 GPU (16 GB VRAM) gücüne sahip olduğu için parametrelerimizi daha verimli hale getirebiliriz:
- `--max-length 512` (Daha geniş bağlam penceresi)
- `--batch-size 4` (Daha hızlı eğitim)
- `--grad-accum 2` (Dengeli adımlar)
- `--token` (Korumalı modele erişim için Hugging Face Token'ınız)

Lütfen aşağıdaki `<YOUR_HF_TOKEN>` kısmını kendi Hugging Face token'ınız ile değiştirip çalıştırın:

In [ ]:
!python peft_finetune.py --epochs 3 --batch-size 4 --grad-accum 2 --max-length 512 --token "<YOUR_HF_TOKEN>"

### 3. Ağırlıkları Birleştirme (Merge) ve Hugging Face'e Yükleme
Hugging Face'in ücretsiz Serverless Inference API'si sadece LoRA adaptörlerini tek başına yükleyip çalıştırmayı desteklemez (`Model not supported by provider hf-inference` hatası verir).

Bu yüzden, aşağıdaki hücreyi çalıştırarak eğittiğimiz LoRA adaptörünü ana Gemma modeliyle birleştirip tek bir bütün model (`Gurkan26/gemma-journal-merged`) haline getireceğiz ve Hugging Face'e yükleyeceğiz:

Lütfen `<YOUR_HF_TOKEN>` değerini yazıp çalıştırın (Birleştirme ve yükleme işlemi 3-5 dakika sürebilir):

In [ ]:
!python merge_and_push.py --token "<YOUR_HF_TOKEN>"